In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "geopandas>1",
#     "lonboard",
#     "palettable",
#     "pandas",
#     "shapely",
#     "sidecar",
#     "pyarrow",
# ]
# ///

# Prix au m² avec Lonboard

Ce carnet charge l'échantillon `valeurs_foncieres_2025_n400k_v2.gpkg` et met en place deux vues Lonboard pour examiner les prix immobiliers au mètre carré : un nuage de points et une carte de chaleur.

In [1]:
from pathlib import Path

import geopandas as gpd
import numpy as np
from IPython.display import display
from lonboard import HeatmapLayer, Map, ScatterplotLayer

DATA_PATH = Path("valeurs_foncieres_2025_n400k_v2.gpkg")

In [2]:
columns = [
    "price_per_sqm",
    "Valeur fonciere",
    "Surface reelle bati",
    "Type local",
    "Commune",
    "Date mutation",
    "Code postal",
    "geometry",
]

gdf = gpd.read_file(DATA_PATH, columns=columns)
if "price_per_sqm" not in gdf or gdf["price_per_sqm"].isna().all():
    surface = gdf["Surface reelle bati"].replace({0: np.nan})
    gdf["price_per_sqm"] = gdf["Valeur fonciere"] / surface

gdf = (
    gdf.dropna(subset=["geometry", "price_per_sqm"])
    .loc[lambda df: df["price_per_sqm"] > 0]
    .to_crs(4326)
)

price_low, price_high = gdf["price_per_sqm"].quantile([0.02, 0.98])
if price_high == price_low:
    price_high = price_low + 1

clipped = gdf["price_per_sqm"].clip(price_low, price_high)
price_scale = (clipped - price_low) / (price_high - price_low)

start_color = np.array([49, 130, 189])
end_color = np.array([178, 24, 43])
colors = (
    start_color + (end_color - start_color) * price_scale.to_numpy()[:, None]
).round().astype(int)
gdf["price_color"] = colors.tolist()

gdf["tooltip_text"] = (
    "Commune : "
    + gdf["Commune"].fillna("N/A")
    + "<br>Type : "
    + gdf["Type local"].fillna("(inconnu)")
    + "<br>Prix au m² : "
    + gdf["price_per_sqm"].round(0).map(lambda v: f"{v:,.0f}".replace(",", " "))
    + " €"
)

price_summary = gdf["price_per_sqm"].describe(percentiles=[0.5, 0.9]).round(2)
price_summary

count      394047.00
mean        13702.71
std        122288.48
min             0.00
50%          2722.22
90%          8333.34
max      27733333.33
Name: price_per_sqm, dtype: float64

## Nuage de points des prix au m²
Chaque point représente une mutation immobilière. La couleur encode le prix (bleu = moins cher, rouge = plus cher) et un tooltip expose les attributs principaux.

In [3]:
scatter_columns = gdf[[
    "Commune",
    "Type local",
    "price_per_sqm",
    "price_color",
    "tooltip_text",
    "geometry",
]].copy()

color_array = np.asarray(scatter_columns["price_color"].tolist(), dtype=np.uint8)

scatter_layer = ScatterplotLayer.from_geopandas(
    scatter_columns.drop(columns=["price_color"]),
    auto_downcast=False,
    get_fill_color=color_array,
    get_radius=8,
    radius_units="pixels",
    radius_min_pixels=2,
    radius_max_pixels=30,
    stroked=False,
    pickable=True,
)

scatter_map = Map(
    scatter_layer,
    height=650,
    show_tooltip=True,
)

try:
    display(scatter_map)
except Exception:
    display(scatter_map.as_html())

## Carte de chaleur des prix moyens
La carte de chaleur agrège les transactions pour faire ressortir les zones où les prix moyens au m² sont les plus élevés.

In [4]:
heatmap_weights = gdf["price_per_sqm"].to_numpy(dtype=float)

heatmap_layer = HeatmapLayer.from_geopandas(
    gdf[["price_per_sqm", "geometry"]],
    auto_downcast=False,
    get_weight=heatmap_weights,
    aggregation="MEAN",
    radius_pixels=60,
    color_domain=(float(price_low), float(price_high)),
)

heatmap_map = Map(
    heatmap_layer,
    height=650,
    show_tooltip=True,
)

try:
    display(heatmap_map)
except Exception:
    display(heatmap_map.as_html())